# 🌿 KISAAN AI — Indian Crops Fine-Tuning
### Adds Rice · Wheat · Cotton · Banana · Mango to existing PlantVillage model
**Prerequisites:**
- Upload your `plant_disease_cnn.h5` to Google Drive before running
- Runtime → Change runtime type → **T4 GPU**
- Expected time: ~20-25 minutes

## Step 1 — Mount Drive + Setup Kaggle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install kaggle -q
print('✅ Drive mounted')

In [ ]:
from google.colab import files
print('Upload your kaggle.json:')
files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print('✅ Kaggle configured')

## Step 2 — Upload your existing model from Drive

In [ ]:
import os, shutil

# ── Option A: copy from Google Drive (recommended) ───────────────────────────
# Put plant_disease_cnn.h5 anywhere in your Drive, update path below
DRIVE_MODEL_PATH = '/content/drive/MyDrive/plant_disease_cnn.h5'

if os.path.exists(DRIVE_MODEL_PATH):
    shutil.copy(DRIVE_MODEL_PATH, '/content/plant_disease_cnn.h5')
    print(f'✅ Model copied from Drive: {os.path.getsize("/content/plant_disease_cnn.h5") / 1e6:.1f} MB')
else:
    print('⚠ Model not found in Drive at:', DRIVE_MODEL_PATH)
    print('Uploading manually instead...')
    files.upload()  # will upload plant_disease_cnn.h5 directly

## Step 3 — Download Indian Crop Datasets from Kaggle

In [ ]:
import os

os.makedirs('/content/indian_data', exist_ok=True)

datasets = [
    # (dataset_slug, local_folder, crop_label)
    ('nirmalsankalana/rice-leaf-disease-image',     'rice_raw',   'rice'),
    ('jaw dadali1045/20k-multi-class-crop-disease-images', 'wheat_raw', 'wheat'),
    ('seroshkarim/cotton-leaf-disease-dataset',    'cotton_raw', 'cotton'),
    ('harshitbana/banana-leaf-disease',            'banana_raw', 'banana'),
    ('warcoder/mango-leaf-disease-dataset',        'mango_raw',  'mango'),
]

for slug, folder, crop in datasets:
    dest = f'/content/indian_data/{folder}'
    if not os.path.exists(dest):
        print(f'Downloading {crop} dataset...')
        os.makedirs(dest, exist_ok=True)
        ret = os.system(f'kaggle datasets download -d "{slug}" -p {dest} --unzip -q 2>&1')
        if ret == 0:
            total = sum(len(f) for _, _, f in os.walk(dest))
            print(f'  ✅ {crop}: {total} files')
        else:
            print(f'  ❌ {crop} download failed — will skip')
    else:
        print(f'  ✅ {crop}: already downloaded')

print('\nAll downloads attempted.')

## Step 4 — Explore Downloaded Structure

In [ ]:
import os

def explore_dir(path, depth=0, max_depth=3):
    if depth > max_depth or not os.path.isdir(path):
        return
    items = sorted(os.listdir(path))
    for item in items[:8]:  # show first 8 only
        full = os.path.join(path, item)
        if os.path.isdir(full):
            n_files = len([f for f in os.listdir(full) if os.path.isfile(os.path.join(full, f))])
            print('  ' * depth + f'📁 {item}/ ({n_files} files)')
            explore_dir(full, depth + 1, max_depth)

print('Downloaded structure:')
explore_dir('/content/indian_data', max_depth=2)

## Step 5 — Build Unified Dataset
This cell walks each downloaded folder, finds image subdirectories,
and copies them into a clean unified structure: `/content/unified/{CropName___ClassName}/`

In [ ]:
import os, shutil
from pathlib import Path

UNIFIED_DIR     = '/content/unified'
PLANTVILLAGE_DIR = None  # will be found automatically
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# ── Find PlantVillage color folder ───────────────────────────────────────────
# PlantVillage was downloaded in previous notebook to /content/plantvillage
# If you don't have it, set SKIP_PLANTVILLAGE = True
SKIP_PLANTVILLAGE = False

for root, dirs, files in os.walk('/content/plantvillage'):
    if 'color' in root.lower() and len(dirs) > 10:
        PLANTVILLAGE_DIR = root
        break
    elif len(dirs) > 20:  # fallback
        PLANTVILLAGE_DIR = root
        break

if PLANTVILLAGE_DIR:
    print(f'Found PlantVillage at: {PLANTVILLAGE_DIR}')
else:
    print('PlantVillage not found — will use Indian crops only (still valid for fine-tuning)')
    SKIP_PLANTVILLAGE = True

os.makedirs(UNIFIED_DIR, exist_ok=True)

def is_image(filepath):
    return Path(filepath).suffix.lower() in IMG_EXTS

def copy_class_folder(src_folder, class_name, count_limit=None):
    """Copy images from src_folder into UNIFIED_DIR/class_name/"""
    dest = os.path.join(UNIFIED_DIR, class_name)
    os.makedirs(dest, exist_ok=True)
    imgs = [f for f in os.listdir(src_folder) if is_image(os.path.join(src_folder, f))]
    if count_limit:
        imgs = imgs[:count_limit]
    for i, img in enumerate(imgs):
        # Rename to avoid collisions across datasets
        ext = Path(img).suffix
        dst_name = f'{class_name}_{i:05d}{ext}'
        shutil.copy2(os.path.join(src_folder, img), os.path.join(dest, dst_name))
    return len(imgs)

total_copied = 0

# ── Copy PlantVillage (all 38 classes) ───────────────────────────────────────
if not SKIP_PLANTVILLAGE and PLANTVILLAGE_DIR:
    print('\nCopying PlantVillage classes...')
    for cls in sorted(os.listdir(PLANTVILLAGE_DIR)):
        src = os.path.join(PLANTVILLAGE_DIR, cls)
        if os.path.isdir(src):
            n = copy_class_folder(src, cls)
            total_copied += n
    print(f'  PlantVillage: {total_copied} images')
else:
    print('Skipping PlantVillage (not available)')

print('\nPlantVillage classes in unified:', len(os.listdir(UNIFIED_DIR)))

In [ ]:
# ── Indian crops: map downloaded folders to unified class names ───────────────
# Format: (raw_folder, unified_class_name)
# We need to inspect actual downloaded folders and map them
# This cell auto-detects subfolder names

import os

CROP_ROOTS = {
    'rice'   : '/content/indian_data/rice_raw',
    'wheat'  : '/content/indian_data/wheat_raw',
    'cotton' : '/content/indian_data/cotton_raw',
    'banana' : '/content/indian_data/banana_raw',
    'mango'  : '/content/indian_data/mango_raw',
}

# Auto-find image class folders in each crop root
def find_class_folders(base_path, crop_name):
    """Walk dir tree to find leaf folders that contain images."""
    class_folders = []
    for root, dirs, files in os.walk(base_path):
        img_files = [f for f in files if Path(f).suffix.lower() in IMG_EXTS]
        if len(img_files) >= 20:  # folder has enough images to be a class
            folder_name = os.path.basename(root)
            # Create unified class name: CropName___ClassName
            unified_name = f'{crop_name.capitalize()}___{folder_name}'
            class_folders.append((root, unified_name, len(img_files)))
    return class_folders

all_indian_classes = []
for crop, root in CROP_ROOTS.items():
    if not os.path.exists(root):
        print(f'⚠ {crop}: folder not found, skipping')
        continue
    classes = find_class_folders(root, crop)
    all_indian_classes.extend(classes)
    print(f'{crop}: {len(classes)} classes found')
    for src, name, count in classes:
        print(f'  → {name} ({count} images)')

print(f'\nTotal Indian crop classes: {len(all_indian_classes)}')

In [ ]:
# ── Copy Indian crop classes into unified dir ────────────────────────────────
# Cap at 800 images per class to balance with PlantVillage
MAX_PER_CLASS = 800

indian_copied = 0
for src_folder, unified_name, total in all_indian_classes:
    n = copy_class_folder(src_folder, unified_name, count_limit=MAX_PER_CLASS)
    indian_copied += n
    print(f'  ✅ {unified_name}: {n} images')

print(f'\nIndian crops: {indian_copied} images added')
print(f'Total unified classes: {len(os.listdir(UNIFIED_DIR))}')

# Count total images
total_imgs = sum(
    len([f for f in os.listdir(os.path.join(UNIFIED_DIR, c)) if is_image(os.path.join(UNIFIED_DIR, c, f))])
    for c in os.listdir(UNIFIED_DIR)
    if os.path.isdir(os.path.join(UNIFIED_DIR, c))
)
print(f'Total images in unified dataset: {total_imgs:,}')

## Step 6 — Load Existing Model + Freeze Base

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import warnings; warnings.filterwarnings('ignore')

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

IMG_SIZE   = 224
BATCH_SIZE = 32

# ── Load existing model ───────────────────────────────────────────────────────
print('\nLoading existing model...')
old_model = keras.models.load_model('/content/plant_disease_cnn.h5')
print(f'Old model loaded — output classes: {old_model.output_shape[-1]}')

# ── Build data generators ─────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.15,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = train_datagen.flow_from_directory(
    UNIFIED_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True, seed=42
)
val_gen = val_datagen.flow_from_directory(
    UNIFIED_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=False, seed=42
)

NUM_CLASSES = len(train_gen.class_indices)
print(f'\nNew total classes: {NUM_CLASSES}')
print(f'Train batches    : {len(train_gen)}')
print(f'Val batches      : {len(val_gen)}')

In [ ]:
# ── Build new model with updated output head ──────────────────────────────────
# Keep MobileNetV2 base + feature layers, replace only the final Dense output

# Extract MobileNetV2 base from old model (first layer after input)
base = None
for layer in old_model.layers:
    if 'mobilenetv2' in layer.name.lower():
        base = layer
        break

if base is None:
    # Fallback: use old model up to second-to-last dense layer as feature extractor
    feature_extractor = keras.Model(
        inputs=old_model.input,
        outputs=old_model.layers[-3].output  # before last dropout + dense
    )
    feature_extractor.trainable = False

    inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = feature_extractor(inputs, training=False)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    new_model = keras.Model(inputs, outputs)
    print('Built model using feature extractor approach')
else:
    base.trainable = False
    inputs  = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    new_model = keras.Model(inputs, outputs)
    print('Built model using MobileNetV2 base approach')

new_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

trainable = sum(tf.size(w).numpy() for w in new_model.trainable_weights)
print(f'Trainable params: {trainable:,}  (head only — base frozen)')
print(f'Output classes  : {NUM_CLASSES}')

## Step 7 — Phase 1: Train New Head (Base Frozen, 5 epochs)

In [ ]:
callbacks_p1 = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
    ModelCheckpoint('/content/best_p1.keras', save_best_only=True, monitor='val_accuracy', verbose=1)
]

print('Phase 1: Training head (base frozen)...')
h1 = new_model.fit(
    train_gen, validation_data=val_gen,
    epochs=5, callbacks=callbacks_p1, verbose=1
)
p1_best = max(h1.history['val_accuracy'])
print(f'\n✅ Phase 1 best val accuracy: {p1_best:.4f} ({p1_best*100:.1f}%)')

## Step 8 — Phase 2: Fine-tune Top Layers (8 epochs)

In [ ]:
# Unfreeze top 30 layers of base for fine-tuning
if base is not None:
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    print(f'Unfrozen: top 30 layers of MobileNetV2')
else:
    feature_extractor.trainable = True
    for layer in feature_extractor.layers[:-15]:
        layer.trainable = False
    print('Unfrozen: top 15 layers of feature extractor')

new_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),  # very low LR
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

callbacks_p2 = [
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, verbose=1),
    ModelCheckpoint('/content/best_p2.keras', save_best_only=True, monitor='val_accuracy', verbose=1)
]

print('Phase 2: Fine-tuning top layers...')
h2 = new_model.fit(
    train_gen, validation_data=val_gen,
    epochs=8, callbacks=callbacks_p2, verbose=1
)
p2_best = max(h2.history['val_accuracy'])
print(f'\n✅ Phase 2 best val accuracy: {p2_best:.4f} ({p2_best*100:.1f}%)')

## Step 9 — Evaluate Final Model

In [ ]:
import matplotlib.pyplot as plt

val_gen.reset()
loss, acc, top3 = new_model.evaluate(val_gen, verbose=1)
print(f'\n=== FINAL RESULTS ===')
print(f'Val Accuracy  : {acc:.4f} ({acc*100:.1f}%)')
print(f'Top-3 Accuracy: {top3:.4f} ({top3*100:.1f}%)')
print(f'Val Loss      : {loss:.4f}')
print(f'Total classes : {NUM_CLASSES}')

# Training curves
all_acc  = h1.history['accuracy']    + h2.history['accuracy']
all_val  = h1.history['val_accuracy']+ h2.history['val_accuracy']
all_loss = h1.history['loss']        + h2.history['loss']
all_vloss= h1.history['val_loss']    + h2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(all_acc,  label='Train', color='#4ade80')
ax1.plot(all_val,  label='Val',   color='#38bdf8')
ax1.axvline(x=len(h1.history['accuracy'])-1, color='gray', linestyle='--', label='Fine-tune start')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(all_loss, label='Train', color='#f97316')
ax2.plot(all_vloss,label='Val',   color='#ef4444')
ax2.axvline(x=len(h1.history['loss'])-1, color='gray', linestyle='--')
ax2.set_title('Loss'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/finetune_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-crop accuracy check for Indian crops
import numpy as np

val_gen.reset()
y_true, y_pred = [], []
for i in range(min(len(val_gen), 50)):  # sample 50 batches
    X_batch, y_batch = val_gen[i]
    preds = new_model.predict(X_batch, verbose=0)
    y_true.extend(np.argmax(y_batch, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

idx_to_class = {v: k for k, v in train_gen.class_indices.items()}
indian_crops  = ['rice', 'wheat', 'cotton', 'banana', 'mango']

print('Per-crop accuracy (Indian crops):')
for crop in indian_crops:
    crop_indices = [i for i, (true,) in enumerate(zip(y_true))
                    if crop.lower() in idx_to_class.get(true, '').lower()]
    if not crop_indices:
        print(f'  {crop:10}: no samples in this batch subset')
        continue
    correct = sum(1 for i in crop_indices if y_true[i] == y_pred[i])
    pct = correct / len(crop_indices) * 100
    print(f'  {crop:10}: {pct:.1f}% ({correct}/{len(crop_indices)})')

## Step 10 — Save Updated Model + Labels

In [ ]:
import json, re
from pathlib import Path

NEW_MODEL_PATH  = '/content/plant_disease_cnn.h5'
NEW_LABELS_PATH = '/content/cnn_class_labels.json'

# Save model — OVERWRITES the original filename so Flask needs no changes
new_model.save(NEW_MODEL_PATH)
print(f'✅ Model saved: {NEW_MODEL_PATH}')

def get_treatment(disease, crop):
    d = disease.lower()
    if 'healthy' in d:
        return 'No treatment needed. Crop looks healthy.'
    elif any(x in d for x in ['blast', 'blight', 'phytophthora']):
        return 'Apply copper-based fungicide. Remove infected leaves. Improve drainage.'
    elif any(x in d for x in ['rust', 'puccinia']):
        return 'Apply triazole fungicide. Avoid overhead irrigation. Use resistant varieties.'
    elif any(x in d for x in ['spot', 'cercospora', 'alternaria', 'brown']):
        return 'Apply mancozeb or chlorothalonil. Ensure proper plant spacing.'
    elif any(x in d for x in ['mildew', 'powdery']):
        return 'Apply sulfur or potassium bicarbonate spray. Improve air circulation.'
    elif any(x in d for x in ['mosaic', 'virus', 'curl', 'leaf roll']):
        return 'No chemical cure. Remove infected plants. Control aphid and whitefly vectors.'
    elif any(x in d for x in ['wilt', 'fusarium', 'verticillium']):
        return 'No cure once infected. Remove plant. Use disease-free seeds next season.'
    elif any(x in d for x in ['anthracnose', 'colletotrichum']):
        return 'Apply copper fungicide or mancozeb. Avoid wetting foliage.'
    elif any(x in d for x in ['rot', 'botrytis', 'sclerotinia']):
        return 'Improve ventilation. Apply iprodione or thiophanate-methyl fungicide.'
    elif 'scab' in d:
        return 'Apply captan or myclobutanil. Prune infected branches. Rake fallen leaves.'
    elif any(x in d for x in ['bacterial', 'xanthomonas', 'pseudomonas']):
        return 'Apply copper bactericide. Avoid overhead irrigation. Remove infected material.'
    elif 'sigatoka' in d or 'panama' in d:
        return 'Apply propiconazole fungicide. Remove severely infected leaves. Improve drainage.'
    else:
        return 'Consult local agronomist. Apply broad-spectrum fungicide as precaution.'

def parse_class(class_name):
    # Handle CropName___DiseaseName OR CropName__DiseaseName
    if '___' in class_name:
        parts = class_name.split('___', 1)
    elif '__' in class_name:
        parts = class_name.split('__', 1)
    else:
        parts = [class_name, 'unknown']
    crop    = parts[0].replace('_', ' ').strip()
    disease = parts[1].replace('_', ' ').strip()
    is_healthy = 'healthy' in disease.lower()
    return {
        'crop'      : crop,
        'disease'   : disease,
        'is_healthy': is_healthy,
        'treatment' : get_treatment(disease, crop)
    }

labels_dict = {
    str(idx): parse_class(class_name)
    for class_name, idx in train_gen.class_indices.items()
}

with open(NEW_LABELS_PATH, 'w') as f:
    json.dump(labels_dict, f, indent=2)

print(f'✅ Labels saved: {NEW_LABELS_PATH}')
print(f'   Total classes: {len(labels_dict)}')

# Show Indian crop classes
print('\nIndian crop entries in labels:')
for k, v in labels_dict.items():
    if v['crop'].lower() in ['rice', 'wheat', 'cotton', 'banana', 'mango']:
        status = '✅' if v['is_healthy'] else '🔴'
        print(f'  {status} [{k}] {v["crop"]} — {v["disease"]}')

## Step 11 — Quick Inference Test

In [ ]:
import random
from tensorflow.keras.preprocessing import image as keras_image

def predict_disease(img_path, top_k=3):
    img = keras_image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = keras_image.img_to_array(img) / 255.0
    arr = np.expand_dims(arr, axis=0)
    preds = new_model.predict(arr, verbose=0)[0]
    top_indices = np.argsort(preds)[::-1][:top_k]
    results = []
    for idx in top_indices:
        info = labels_dict[str(idx)]
        results.append({
            'crop'      : info['crop'],
            'disease'   : info['disease'],
            'is_healthy': info['is_healthy'],
            'confidence': round(float(preds[idx]) * 100, 2)
        })
    return results

# Test with one image from each Indian crop
for crop in ['rice', 'wheat', 'cotton', 'banana', 'mango']:
    crop_classes = [c for c in os.listdir(UNIFIED_DIR) if crop.lower() in c.lower()]
    if not crop_classes:
        print(f'{crop}: no classes found in unified dir, skipping')
        continue
    cls_dir = os.path.join(UNIFIED_DIR, random.choice(crop_classes))
    imgs = [f for f in os.listdir(cls_dir) if is_image(os.path.join(cls_dir, f))]
    if not imgs:
        continue
    test_img = os.path.join(cls_dir, random.choice(imgs))
    result = predict_disease(test_img)
    print(f'{crop.upper()}: {result[0]["crop"]} — {result[0]["disease"]} ({result[0]["confidence"]}%)')

## Step 12 — Download Updated Files

In [ ]:
from google.colab import files

size_mb = os.path.getsize(NEW_MODEL_PATH) / (1024 * 1024)
print(f'Model size: {size_mb:.1f} MB')
print(f'Classes   : {len(labels_dict)}')
print('\nDownloading...')

files.download(NEW_MODEL_PATH)
files.download(NEW_LABELS_PATH)

print('\n✅ Done!')
print('Replace both files in backend/models/ — Flask picks up changes on restart.')
print('No changes needed to app.py or dashboard.html.')

## Summary
After running all cells:
- `plant_disease_cnn.h5` — updated model covering 19 crops (~55 classes)
- `cnn_class_labels.json` — updated labels with Indian crop diseases + treatments

**New crops added:** Rice · Wheat · Cotton · Banana · Mango

**How to deploy:** Replace both files in `backend/models/` and restart Flask with `python app.py`.
The `/predict/image` endpoint works identically — zero code changes needed.